[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# *pandas* 3 esencial.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo I: *Datos y su calidad*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se presentarán los cambios de *pandas* 3 que afectan al resto del curso y se repasarán las operaciones esenciales para cargar y explorar un *dataframe*, usando el *dataset* `penguins`.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencia de la API: curso py311.

Este curso no repasa la API método por método: esa explicación detallada está en los *notebooks* del curso [py311](https://github.com/PythonistaMX/py311). Aquí se hace un repaso breve y el tiempo se dedica a las **decisiones de análisis**: qué tipo de dato elegir, qué hacer con un faltante y qué consecuencias tiene cada decisión. Para esta sesión:

| Tema | *Notebooks* de py311 |
|---|---|
| *NumPy* | [02 Conceptos básicos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/02_conceptos_basicos_de_numpy.ipynb) · [03 Gestión de arreglos](https://github.com/PythonistaMX/py311/blob/main/03_gestion_de_arreglos_de_numpy.ipynb) · [04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) · [05 Operaciones básicas con arreglos](https://github.com/PythonistaMX/py311/blob/main/05_operaciones_basicas_con_arreglos.ipynb) |
| *pandas* | [09 Introducción a *pandas*](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb) · [10 Tipos de datos](https://github.com/PythonistaMX/py311/blob/main/10_tipos_de_datos_de_pandas.ipynb) · [11 Operaciones básicas con *dataframes*](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) · [12 Índices y multiíndices](https://github.com/PythonistaMX/py311/blob/main/12_indices_y_multiindices.ipynb) · [13 Datos categóricos](https://github.com/PythonistaMX/py311/blob/main/13_datos_categoricos.ipynb) |

> ⚠️ **Nota: py311 usa *pandas* 2.** Sus *notebooks* se escribieron para *pandas* 2.x y este curso usa *pandas* 3, así que algunos resultados se verán distintos. Por ejemplo, allá el texto aparece con `dtype` `object` (aquí es `str`), se explica el parámetro `inplace` y la asignación encadenada como si modificaran el original (con *pandas* 3 ya no lo hacen sobre columnas o selecciones) y puede aparecer `SettingWithCopyWarning`, que ya no existe. La sección 1.1 explica estos cambios. Cuando haya diferencias, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

Se importarán las bibliotecas con sus alias convencionales, se ubicará la carpeta `data/` (que ya trae los CSV del curso, sin necesidad de internet) y se fijará la semilla `401` para que los resultados aleatorios sean reproducibles.

* La siguiente celda importará las bibliotecas y definirá `DATA` y `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda mostrará las versiones instaladas. El curso usa *pandas* 3.

In [ ]:
{"NumPy": np.__version__, "pandas": pd.__version__, "seaborn": sns.__version__}

---
## 1. *pandas* esencial (35 min).

[*pandas*](https://pandas.pydata.org) agrega a *NumPy* **etiquetas** (índice y nombres de columna) y **columnas de tipos distintos**:

* `Series`: una columna (arreglo 1D + índice).
* `DataFrame`: una tabla en la que cada columna es una `Series` con su propio `dtype`.

### 1.1 *pandas* 3: una gran actualización de arquitectura.

El curso usa **[*pandas* 3.0](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html)** (publicado el 21 de enero de 2026). Es la versión mayor más importante en años y cambia dos cosas de fondo:

| Cambio | *pandas* 2.x | *pandas* 3 |
|---|---|---|
| **Copy-on-Write (CoW)** | Una selección podía ser una vista o una copia, según la operación | **Obligatorio:** todo resultado de una selección o de un método **se comporta como una copia**. La única forma de modificar un objeto es modificarlo directamente (por ejemplo, con `df.loc[...] = ...`). |
| **Tipo de texto por defecto** | `object` (cualquier objeto de *Python*) | **`str`**, un tipo dedicado a texto **respaldado por *PyArrow*** cuando está instalado (en el curso lo está). Solo admite texto o faltantes, y el faltante es siempre `NaN`. |

Además: `SettingWithCopyWarning` desaparece, las fechas leídas de texto se guardan por defecto con resolución de microsegundos (`datetime64[us]`), `groupby` usa `observed=True` por defecto con columnas categóricas y se eliminaron muchas funciones que ya estaban obsoletas. Se requiere *Python* ≥ 3.11 y *NumPy* ≥ 1.26.

> ⚠️ **ADVERTENCIA: incompatibilidad hacia atrás.**
>
> El código escrito para *pandas* 2.x **puede cambiar de comportamiento en silencio o fallar** con *pandas* 3. Los casos más comunes:
> 1. **Asignación encadenada** (`df["col"][i] = valor`, `df[mascara]["col"] = valor`): **ya nunca modifica** el *dataframe* original.
> 2. **Métodos con `inplace=True` sobre una columna** (`df["col"].fillna(0, inplace=True)`): **tampoco modifican** el original.
> 3. **Comprobaciones de tipo de texto** como `df["col"].dtype == object` o `select_dtypes(include="object")`: el texto ahora es `str`.
> 4. **Asignar un valor que no es texto** a una columna `str` (por ejemplo, un número): **lanza `TypeError`**.
>
> Los tutoriales, respuestas de foros y código generado antes de 2026 suelen asumir *pandas* 2. Las celdas siguientes ejecutan cada caso. Para migrar código propio, la recomendación oficial es pasar primero a *pandas* 2.3, eliminar todas sus advertencias y después actualizar ([guía de migración del tipo `str`](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html), [guía de *Copy-on-Write*](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)).

* La siguiente celda creará un *dataframe* pequeño para los ejemplos.

In [ ]:
ejemplo = pd.DataFrame({"especie": ["Adelie", "Gentoo", "Chinstrap"],
                        "masa": [3750.0, np.nan, 3700.0]})
ejemplo.dtypes

**Ejemplo 1: el texto ya no es `object`.** La columna `especie` es de tipo `str` y está respaldada por *PyArrow*; la comparación con `object` que funcionaba en *pandas* 2 ahora devuelve `False`.

* La siguiente celda comparará el `dtype` de la columna de texto con `object` y con la forma recomendada.

In [ ]:
{
    "dtype": repr(ejemplo["especie"].dtype),
    "arreglo interno": type(ejemplo["especie"].array).__name__,
    "dtype == object (estilo pandas 2)": ejemplo["especie"].dtype == object,
    "pd.api.types.is_string_dtype (recomendado)": pd.api.types.is_string_dtype(ejemplo["especie"]),
}

**Ejemplo 2: asignación encadenada.** En *pandas* 2 a veces modificaba el original (con `SettingWithCopyWarning`); en *pandas* 3 **nunca** lo hace y emite `ChainedAssignmentError`. La celda captura la advertencia para mostrarla.

* La siguiente celda intentará cambiar la masa del primer pingüino con asignación encadenada.

In [ ]:
copia = ejemplo.copy()
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    copia["masa"][0] = 9999
copia.loc[0, "masa"], [type(a.message).__name__ for a in avisos]

* La siguiente celda hará la misma asignación de la forma correcta, en un solo paso con `loc`.

In [ ]:
copia.loc[0, "masa"] = 9999
copia.loc[0, "masa"]

**Ejemplo 3: `inplace=True` sobre una columna.** El método actúa sobre una copia intermedia, así que el faltante sigue ahí.

* La siguiente celda intentará rellenar el faltante con `inplace=True` sobre la columna.

In [ ]:
copia = ejemplo.copy()
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    copia["masa"].fillna(0, inplace=True)
copia["masa"].isna().sum(), [type(a.message).__name__ for a in avisos]

* La siguiente celda mostrará la forma recomendada: reasignar la columna.

In [ ]:
copia["masa"] = copia["masa"].fillna(0)
copia["masa"].isna().sum()

**Ejemplo 4: una columna `str` solo admite texto.** En *pandas* 2 la columna `object` aceptaba cualquier objeto; ahora la asignación falla.

* La siguiente celda intentará guardar un número en la columna de texto y mostrará el error.

In [ ]:
try:
    copia.loc[0, "especie"] = 5
    resultado = "se asignó"
except TypeError as error:
    resultado = f"TypeError: {error}"
resultado

**Ejemplo 5: una selección se comporta como copia.** Modificar `serie` no altera `copia`, aunque por dentro *pandas* comparta memoria hasta el momento de escribir (*copy-on-write*).

* La siguiente celda modificará una columna extraída y comprobará que el *dataframe* no cambió.

In [ ]:
copia = ejemplo.copy()
serie = copia["masa"]
serie.iloc[2] = 0.0
serie.iloc[2], copia.loc[2, "masa"]

**Regla práctica para todo el curso:** para modificar un *dataframe* se usa **`df.loc[filas, columnas] = valor`** o se **reasigna** (`df["col"] = ...`, `df = df.assign(...)`). Para revisar si una columna es texto se usa `pd.api.types.is_string_dtype`.

### 1.2 Repaso exprés de *pandas*.

| Tarea | Lo esencial | En py311 |
|---|---|---|
| Estructuras | `Series` (una columna con índice) y `DataFrame` (columnas con `dtype` propio) | [09](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb) |
| Primer vistazo | `read_csv`, `head`, `shape`, `info`, `describe`, `value_counts` | [09](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb), [11](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) |
| Selección | `df["col"]`; `loc` por **etiqueta** (`a:b` incluye `b`); `iloc` por **posición** (`a:b` excluye `b`) | [09](https://github.com/PythonistaMX/py311/blob/main/09_introduccion_a_pandas.ipynb), [12](https://github.com/PythonistaMX/py311/blob/main/12_indices_y_multiindices.ipynb) |
| Filtros | Máscaras booleanas con `&`, `\|`, `~`; `isin`; `query` | [11](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) |
| Nuevas columnas | `assign` dentro de una cadena de métodos | [11](https://github.com/PythonistaMX/py311/blob/main/11_operaciones_basicas_con_dataframes.ipynb) |

### 1.3 El *dataset* `penguins` y primer vistazo.

**Palmer Penguins** (Gorman, Williams y Fraser, 2014): medidas de 344 pingüinos de tres especies en tres islas del archipiélago Palmer (Antártida), registradas en las temporadas 2007 a 2009 por el programa *Palmer Station LTER*. Es el *dataset* principal de las sesiones 1 a 6.

| Columna | Descripción |
|---|---|
| `species` | Especie: Adelie, Chinstrap, Gentoo |
| `island` | Isla: Biscoe, Dream, Torgersen |
| `bill_length_mm` | Largo del pico (culmen), mm |
| `bill_depth_mm` | Profundidad del pico, mm |
| `flipper_length_mm` | Largo de la aleta, mm |
| `body_mass_g` | Masa corporal, g |
| `sex` | Sexo (en este CSV: `MALE`/`FEMALE`) |

* La siguiente celda leerá `penguins.csv` y mostrará las primeras filas.

In [ ]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
pinguinos.head()

**Interpretación:** cada fila es un pingüino y cada columna una variable. Ya en las primeras filas aparecen dos cosas que se atenderán: una fila con todas las medidas faltantes (la de índice 3) y el sexo escrito en mayúsculas (`MALE`/`FEMALE`), que se normalizará en la sesión 2.

**Nota:** `info()` es lo primero que hay que revisar: tipos, conteo de no nulos y memoria. Observa que las columnas de texto aparecen como `str`, no como `object`.

**¿Por qué este paso?** `info()` responde en una sola salida tres preguntas de la auditoría: cuántas filas hay, de qué tipo es cada columna y cuántos valores **no nulos** tiene cada una. Es lo primero que se ejecuta con cualquier *dataset* nuevo.

* La siguiente celda mostrará el resumen de `info()`.

In [ ]:
pinguinos.info()

**Interpretación:** hay 344 filas; las medidas tienen 342 valores no nulos (faltan 2) y `sex` tiene 333 (faltan 11). Las columnas de texto son `str` y las medidas, `float64`, incluso las que se anotaron en enteros, por los faltantes.

* La siguiente celda mostrará `describe(include="all")`, que también resume las columnas de texto.

In [ ]:
pinguinos.describe(include="all").T

**Interpretación:** para las numéricas, `describe` da conteo, media, desviación estándar, mínimo, cuartiles y máximo; para las de texto, el número de valores distintos (`unique`), el más frecuente (`top`) y su frecuencia (`freq`). La masa va de 2,700 a 6,300 g, un rango plausible, y su media (4,202 g) supera a la mediana (4,050 g), una primera pista de asimetría que se estudiará en la sesión 3.

* La siguiente celda repasará la selección: las mismas dos filas con `loc` (etiquetas, el extremo se incluye) y con `iloc` (posiciones, el extremo se excluye).

In [ ]:
pinguinos.loc[0:1, ["species", "body_mass_g"]], pinguinos.iloc[0:2, [0, 5]]

* La siguiente celda repasará los filtros: una máscara combinada, `isin` y `query`.

In [ ]:
gentoo_pesados = pinguinos[(pinguinos["species"] == "Gentoo") & (pinguinos["body_mass_g"] > 5500)]
dos_islas = pinguinos[pinguinos["island"].isin(["Dream", "Torgersen"])]
len(gentoo_pesados), len(dos_islas), len(pinguinos.query("species == 'Chinstrap' and flipper_length_mm >= 205"))

### 1.4 Crear columnas con `assign` y encadenar métodos.

`assign` devuelve un **nuevo** *dataframe*, sin modificar el original, y permite escribir el análisis como una receta de pasos encadenados. Con *Copy-on-Write* este estilo es eficiente: *pandas* evita copias innecesarias.

* La siguiente celda creará `body_mass_kg`, filtrará Biscoe, elegirá columnas y ordenará por masa.

In [ ]:
(
    pinguinos
    .assign(body_mass_kg=lambda d: d["body_mass_g"] / 1000)
    .query("island == 'Biscoe'")
    .loc[:, ["species", "sex", "body_mass_kg"]]
    .sort_values("body_mass_kg", ascending=False)
    .head()
)

**Nota:** *pandas* 3 introduce además `pd.col("columna")` para escribir expresiones sin `lambda`, por ejemplo `assign(body_mass_kg=pd.col("body_mass_g") / 1000)`. En el curso se usará `lambda` porque funciona en cualquier versión y es lo que se encuentra en la mayoría de la documentación.

### 1.5 Decisiones al explorar: *pandas* ignora los faltantes en silencio.

Casi todos los resúmenes de *pandas* (`mean`, `std`, `value_counts`, `describe`) **excluyen los faltantes sin avisar**. Es cómodo, pero cada columna termina con su propio *n* y las proporciones se calculan sobre un denominador que no siempre es el que se cree. La decisión es **qué denominador reportar** y dejarlo explícito.

**Concepto: el denominador.** Una proporción es $\hat{p} = k / n$. Si los faltantes se excluyen, $n$ es el número de valores **con dato**; si se incluyen, es el total de filas. Ambas son legítimas, pero responden preguntas distintas.

* La siguiente celda comparará el número de filas con el *n* real de cada columna.

In [ ]:
pinguinos.count().to_frame("n_con_dato").assign(n_filas=len(pinguinos), faltan=lambda d: d["n_filas"] - d["n_con_dato"])

* La siguiente celda calculará la proporción de cada sexo sin contar los faltantes (por defecto) y contándolos.

In [ ]:
pd.DataFrame({
    "sin faltantes (por defecto)": pinguinos["sex"].value_counts(normalize=True),
    "con faltantes": pinguinos["sex"].value_counts(normalize=True, dropna=False),
}).round(3)

Si se reporta "50.5% son machos", hay que decir que es **entre los 333 con sexo registrado**: entre los 344 pingüinos la cifra es 48.8%. Ninguna es incorrecta; es incorrecto no decir cuál se usa.

### 🟢 Ejercicio 1 · Explorar con *pandas*.

1. ¿Cuántos pingüinos hay de cada especie **en cada isla**? (pista: `pd.crosstab`). ¿Qué especie vive en las tres islas?
2. Con `loc`, obtén `species`, `island` y `body_mass_g` de los Gentoo con masa mayor a 5,000 g, ordenados de mayor a menor masa. ¿Cuántos son?

**🎯 Objetivo:** practicar tablas cruzadas y filtros combinados con `loc`. **Qué se espera aprender:** a describir **cómo se reparten** las observaciones entre grupos, la base de cualquier comparación posterior.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

* La siguiente celda es para tu solución del inciso 2.

In [ ]:
# TU CÓDIGO AQUÍ (inciso 2)

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 2 · Nuevas variables con `assign`.

Crea, en un solo `assign`, las columnas `bill_ratio = bill_length_mm / bill_depth_mm` y `body_mass_kg`. Muestra los 5 pingüinos con mayor `bill_ratio`. ¿De qué especie son? ¿Por qué crees que pasa?

**🎯 Objetivo:** crear variables **derivadas** dentro de una cadena de métodos. **Qué se espera aprender:** que una variable derivada (un cociente) puede capturar algo que las originales no muestran por separado, y que su fórmula debe documentarse.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Asignación encadenada en *pandas* 3.

El siguiente código intenta poner en `NaN` la masa de los pingüinos de Torgersen. Con *pandas* 3 **no modifica `copia`** y emite `ChainedAssignmentError`. Explica por qué y corrígelo con `loc` en una sola instrucción. Comprueba el resultado contando los faltantes por isla.

```python
copia = pinguinos.copy()
copia[copia["island"] == "Torgersen"]["body_mass_g"] = np.nan
```

**🎯 Objetivo:** diagnosticar y corregir un patrón de *pandas* 2 que en *pandas* 3 deja de funcionar. **Qué se espera aprender:** a reconocer la asignación encadenada en código ajeno (tutoriales, respuestas de foros) y a reescribirla con `loc`.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

**💡 Ideas clave de la sección 1:**
* *pandas* 3 cambió de arquitectura: *Copy-on-Write* obligatorio y texto como `str`. Para modificar se usa `df.loc[...] = ...` o se reasigna.
* `head`, `info` y `describe` son el primer vistazo obligatorio de cualquier *dataset*.
* `loc` selecciona por etiqueta (incluye el extremo), `iloc` por posición (lo excluye).
* *pandas* ignora los faltantes en silencio: hay que reportar el denominador.

---
## ✅ Cierre.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `04_tipos_de_variables.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>